Embedding模型和RAG检索

In [34]:
from zai import ZhipuAiClient
from dotenv import load_dotenv
from pydantic import BaseModel, ConfigDict
from enum import Enum
import numpy as np
import pandas as pd

load_dotenv()
client = ZhipuAiClient()

# 强制所有列名左对齐（默认数字右对齐，文本左对齐，统一后视觉更整齐）
pd.set_option('display.colheader_justify', 'left')
# 【关键】如果数据包含中文，必须开启此项，否则中文字符宽度计算错误会导致严重错位
pd.set_option('display.unicode.east_asian_width', True)
# 取消整体显示宽度限制，防止表格被强行换行破坏结构
pd.set_option('display.width', None) 

class Usage(BaseModel):
    model_config = ConfigDict(from_attributes=True)
    prompt_tokens: int = 0
    completion_tokens: int = 0
    cache_tokens: int = 0
    total_tokens: int = 0
    @property
    def output(self) -> str:
        return f"提示词:{self.prompt_tokens} 回复内容:{self.completion_tokens} 缓存:{self.cache_tokens} 共计:{self.total_tokens}"

class EmbeddingData(BaseModel):
    index: int
    embedding: list[float]
    # ✅ 允许从 SDK 的 Embedding 对象中提取属性
    model_config = ConfigDict(from_attributes=True) 

class EmbeddingResponse(BaseModel):
    model: str
    data: list[EmbeddingData]
    usage: Usage
    # ✅ 允许从已有对象的属性中提取数据
    model_config = ConfigDict(from_attributes=True) 

def embeddings(inputs:list[str], dimensions: int = 2048) -> list[EmbeddingData]:
    """对一组文本向量化"""
    response = client.embeddings.create(
        model="embedding-3",
        input=inputs,
        dimensions=dimensions
    )
    model = EmbeddingResponse.model_validate(response)
    return model.data

def calculate_similarity(a_embedding: list[float], b_embedding: list[float]) -> float:
    """计算两个向量的相似度"""
    # 0. 生成向量
    vec_a = np.array(a_embedding)
    vec_b = np.array(b_embedding)

    # 1. 计算 点积
    dot_product = np.dot(vec_a, vec_b)

    # 3. 计算 模长
    norm_a = np.linalg.norm(vec_a)
    norm_b = np.linalg.norm(vec_b)

    # 4. 相似度
    similarity = dot_product / (norm_a * norm_b)

    return similarity

In [35]:
sentences = [
    "我今天去公园散步了", # A
    "我在公园走了走,很惬意", # A' (和A语义近,但用词不同)
    "Swift 是一门编程语言", # B (和A完全无关)
    "苹果公司开发了 Swift", # B' (和B相关)
    "苹果公司开发了 OC语言",
    "Swift语言是由苹果公司开发",
    "苹果公司开发的Swift语言非常安全",
]
embedding_messages: list[EmbeddingData] = []

def vector(inputs: list[str]) -> list[EmbeddingData]:
    return embeddings(inputs=inputs)

def search(prompt: str, top_k: int = 3) -> list[tuple[str, float]]:
    """语义检索:返回最相似的 top_k 条 (文本, 相似度分数)"""
    # 先生成待检索的向量
    prompt_vec = embeddings(inputs=[prompt])[0].embedding
    # 和“向量数据库”进行比较相似度，生成相似度列表
    scores: list[tuple[str, float]] = [
        (sentences[e.index], calculate_similarity(prompt_vec, e.embedding))
        for e in embedding_messages
    ]
    # 按相似度降序,取前 top_k
    scores.sort(key=lambda x: x[1], reverse=True)
    return scores[:top_k]


In [41]:
    # 生成向量数据库
    global embedding_messages
    embedding_messages = vector(inputs=sentences)
    print(f"向量数据库: {len(embedding_messages)}")

    # 比较两个向量的相似度
    n = len(sentences)
    matrixs: list[list[float]] = []
    for i in range(n):
        temp_s: list = []
        for j in range(n):
            s = calculate_similarity(embedding_messages[i].embedding,
                                     embedding_messages[j].embedding)
            temp_s.append(s)
        matrixs.append(temp_s)

    df = pd.DataFrame(data=matrixs, index=sentences, columns=range(n))
    print(df.round(3))

    # RAG语意检索
    texts = search(prompt="哪门语言是苹果做的?")
    print(f"\nRAG检索出结果：{texts}")

向量数据库: 7
                        0      1      2      3      4      5      6
我今天去公园散步了           1.000  0.878  0.482  0.449  0.437  0.444  0.410
我在公园走了走,很惬意         0.878  1.000  0.485  0.458  0.455  0.453  0.452
Swift 是一门编程语言       0.482  0.485  1.000  0.868  0.687  0.892  0.779
苹果公司开发了 Swift       0.449  0.458  0.868  1.000  0.809  0.926  0.841
苹果公司开发了 OC语言        0.437  0.455  0.687  0.809  1.000  0.786  0.724
Swift语言是由苹果公司开发     0.444  0.453  0.892  0.926  0.786  1.000  0.852
苹果公司开发的Swift语言非常安全  0.410  0.452  0.779  0.841  0.724  0.852  1.000

RAG检索出结果：[('苹果公司开发了 OC语言', np.float64(0.6909542748077965)), ('Swift语言是由苹果公司开发', np.float64(0.6801719241525434)), ('苹果公司开发了 Swift', np.float64(0.6413475689371322))]


对矩阵计算进行优化

In [45]:
# 假设 embedding_messages 是你的模型返回结果列表
embeddings = [msg.embedding for msg in embedding_messages]

# 【优化点】使用 NumPy 一次性将所有向量转为矩阵 (形状为 n x dim)
embedding_matrix = np.array(embeddings)

# 【优化点】利用 NumPy 的矩阵乘法一步算出所有余弦相似度
# 1. 对每一行求 L2 范数(模长)，并保持维度以便广播
norms = np.linalg.norm(embedding_matrix, axis=1, keepdims=True)
# 2. 归一化向量 (防止除以0，加上极小值)
normalized_embeddings = embedding_matrix / (norms + 1e-8)
# 3. 矩阵相乘得出完整的相似度矩阵 (形状为 n x n)
similarity_matrix = np.dot(normalized_embeddings, normalized_embeddings.T)

# 【修复点】直接使用计算好的二维矩阵生成 DataFrame
df = pd.DataFrame(
    data=similarity_matrix, 
    index=sentences, 
    columns=[s[:3] for s in sentences]
)

print(df.round(3))

                      我今天    我在公    Swi    苹果公    苹果公    Swi    苹果公
我今天去公园散步了           1.000  0.878  0.482  0.449  0.437  0.444  0.410
我在公园走了走,很惬意         0.878  1.000  0.485  0.458  0.455  0.453  0.452
Swift 是一门编程语言       0.482  0.485  1.000  0.868  0.687  0.892  0.779
苹果公司开发了 Swift       0.449  0.458  0.868  1.000  0.809  0.926  0.841
苹果公司开发了 OC语言        0.437  0.455  0.687  0.809  1.000  0.786  0.724
Swift语言是由苹果公司开发     0.444  0.453  0.892  0.926  0.786  1.000  0.852
苹果公司开发的Swift语言非常安全  0.410  0.452  0.779  0.841  0.724  0.852  1.000
